# Independent climatology arithmetic review

**Owner: Cameron (Econ/Stats). Reviewer: Mason.**

This is a sample-only statistical cross-check using the prepared
`data/sample/county_windows_sample.csv`, not a production baseline or model fit.
Compute the unsmoothed frequency of positive labels for each observed county-month
in the configured training years. Use Python's standard-library counters and
fractions, independently of any production model implementation.

The denominator is all observed training windows in a group, including negatives.
An unobserved county-month has **no estimate**, not probability zero. This sparse,
synthetic fixture cannot establish historical climatology or complete coverage.
Smoothing and unseen-group fallback remain undecided in `configs/model.yaml`;
this review does not choose them. Historical rates used as training features also
require past-only construction and publication-lag checks, beyond this arithmetic.


In [ ]:
import csv
import hashlib
import json
from collections import Counter
from datetime import datetime, timedelta
from fractions import Fraction

from IPython.display import Markdown, display

from stormroute.config import REPO_ROOT, data_path, load_config

MODEL = load_config("model")
SOURCE = data_path("sample") / "county_windows_sample.csv"
with SOURCE.open(encoding="utf-8", newline="") as handle:
    prepared_rows = list(csv.DictReader(handle))
train_years = set(MODEL["splits"]["train"]["years"])
print("Input:", SOURCE.relative_to(REPO_ROOT))
print("Mode: synthetic sample only; no historical estimates")
print("Training years:", sorted(train_years))

## Calculation

**Question.** Do independently counted positives and denominators reproduce the
prepared fixture's expected county-month frequencies without held-out labels?

Validate the declared split against UTC year before counting. Selection,
calibration, and test labels are never parsed by the calculation. Reject duplicate
training county-window keys, malformed labels, and misaligned six-hour windows.


In [ ]:
def independent_counts(rows):
    """Count observed training labels without reading held-out target values."""
    totals = Counter()
    positives = Counter()
    excluded = Counter()
    seen = set()
    split_years = {
        name: set(spec["years"]) for name, spec in MODEL["splits"].items() if isinstance(spec, dict)
    }
    for row in rows:
        stamp = datetime.fromisoformat(row["window_start_utc"].replace("Z", "+00:00"))
        if stamp.utcoffset() != timedelta(0):
            raise ValueError("Window start must be timezone-aware UTC")
        split = row["split"]
        if split not in split_years or stamp.year not in split_years[split]:
            raise ValueError("Split disagrees with configured calendar years")
        if split != "train":
            excluded[split] += 1
            continue
        county = row["county_fips"]
        if len(county) != 5 or not county.isdigit() or not county.startswith("37"):
            raise ValueError("Expected a five-character NC county FIPS")
        end = datetime.fromisoformat(row["window_end_utc"].replace("Z", "+00:00"))
        if end.utcoffset() != timedelta(0) or end - stamp != timedelta(hours=6):
            raise ValueError("Expected a six-hour UTC window")
        if stamp.hour % 6 or stamp.minute or stamp.second or stamp.microsecond:
            raise ValueError("Window is not aligned to a six-hour boundary")
        key = (county, stamp)
        if key in seen:
            raise ValueError("Duplicate training county-window")
        seen.add(key)
        label = row["label_flood_event"]
        if label not in {"True", "False"}:
            raise ValueError("Expected an explicit boolean label")
        group = (county, stamp.month)
        totals[group] += 1
        positives[group] += int(label == "True")
    result = []
    for county, month in sorted(totals):
        group = (county, month)
        fraction = Fraction(positives[group], totals[group])
        result.append(
            {
                "county_fips": county,
                "month": month,
                "windows": totals[group],
                "positives": positives[group],
                "negatives": totals[group] - positives[group],
                "exact_fraction": str(fraction),
                "unsmoothed_rate": float(fraction),
            }
        )
    return result, dict(excluded)


review, excluded = independent_counts(prepared_rows)
headers = ["County FIPS", "Month", "Windows", "Positives", "Negatives", "Exact rate"]
lines = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
for row in review:
    lines.append(
        "| "
        + " | ".join(
            str(row[key])
            for key in [
                "county_fips",
                "month",
                "windows",
                "positives",
                "negatives",
                "exact_fraction",
            ]
        )
        + " |"
    )
display(Markdown("\n".join(lines)))
print("Excluded row counts by split:", excluded)

## Independent expected values and isolation checks

The expected fractions below are hand-counted from this version of the fixture.
They are test expectations, not claims about North Carolina. A held-out-label
mutation must not change the result, and duplicate training records must fail.


In [ ]:
expected = {("37001", 9): (1, 1), ("37083", 6): (1, 2), ("37087", 12): (1, 1), ("37129", 7): (0, 1)}
actual = {(r["county_fips"], r["month"]): (r["positives"], r["windows"]) for r in review}
assert actual == expected, "Review fixture changes before updating expectations"
mutated = [dict(row) for row in prepared_rows]
for row in mutated:
    if row["split"] != "train":
        row["label_flood_event"] = "HELD_OUT_LABEL_MUST_NOT_BE_READ"
assert independent_counts(mutated) == (review, excluded)
training_row = next(row for row in prepared_rows if row["split"] == "train")
try:
    independent_counts([training_row, training_row])
except ValueError as error:
    assert "Duplicate" in str(error)
else:
    raise AssertionError("Duplicate window was accepted")
assert independent_counts([]) == ([], {})
print("Hand-counted fractions, held-out isolation, duplicate rejection, and empty input passed.")

In [ ]:
output = REPO_ROOT / "outputs/metrics/sample/independent_climatology_review.json"
output.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "status": "Sample arithmetic verified; full-data review and smoothing pending",
    "owner": "Cameron (Econ/Stats)",
    "reviewer": "Mason",
    "source_file": str(SOURCE.relative_to(REPO_ROOT)),
    "source_sha256": hashlib.sha256(SOURCE.read_bytes()).hexdigest(),
    "model_config_sha256": hashlib.sha256(
        (REPO_ROOT / "configs/model.yaml").read_bytes()
    ).hexdigest(),
    "train_years": sorted(train_years),
    "excluded_rows_by_split": excluded,
    "groups": review,
    "unobserved_group_policy": "No estimate; never fill with zero",
}
output.write_text(json.dumps(payload, indent=2, allow_nan=False) + "\n", encoding="utf-8")
print("Saved", output.relative_to(REPO_ROOT))
display(
    Markdown(
        f"**Conclusion.** Independently verified {sum(r['windows'] for r in review)} training "
        f"windows and {sum(r['positives'] for r in review)} positives across {len(review)} "
        "county-month groups. Held-out labels do not affect the calculation. "
        "Full-data verification requires Mason's prepared county-window table; "
        "smoothing, unseen-group fallback, and coverage validation remain pending."
    )
)